# 01 - Data Screening

**Notebook version:** v25 -- 2026-08-02

Load raw SECOM data, inspect missingness/variance, and build the data dictionary.

The first call to `load_raw()` below will automatically download the raw SECOM
data from the UCI Machine Learning Repository into `data/raw/` if it isn't
already present -- no separate script needs to be run first. This requires
internet access and may take a few seconds the first time.

In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import pandas as pd
from preprocessing import load_raw, screen_missingness, screen_variance

X, y = load_raw()
print(f"Shape: {X.shape}, labels: {y.shape}")
y.value_counts()


## Raw DataFrame shape and dtypes (screenshot this cell's output)

This is the exact cell the synopsis's Data Description section refers to
for the required screenshot -- confirms the loaded 1,567 x 590 raw SECOM
DataFrame's shape and column dtypes directly, not just a printed shape
string.

In [ ]:
X.info()


In [ ]:
X.head()


## Missingness and variance profile

Fill these numbers into `docs/data_dictionary_template.csv`.

In [ ]:
missing_frac = X.isna().mean().sort_values(ascending=False)
missing_frac.head(20)

In [ ]:
variances = X.var(numeric_only=True).sort_values()
variances.head(20)

In [ ]:
corr_with_target = X.corrwith(y.astype(float)).abs().sort_values(ascending=False)
corr_with_target.head(20)

## Apply screening thresholds

Default thresholds: drop columns with >40% missing values or near-zero variance.
Adjust and document the final thresholds used in the capstone report.

In [ ]:
X_screened = screen_missingness(X, max_missing_frac=0.4)
X_screened = screen_variance(X_screened, min_variance=1e-6)
print(f"Remaining features: {X_screened.shape[1]} (from {X.shape[1]})")

## Populate data/model_ready/

**Real bug, found after a full pipeline run**: `data/model_ready/`
documented itself as holding cleaned/screened data, but nothing in any
notebook ever actually called `run_screening_pipeline()` -- it only ran
under `if __name__ == "__main__"`, reachable only by executing
`python src/preprocessing.py` directly, which nothing in this pipeline
does. The folder was architecturally orphaned, not just "not yet run."
Fixed by calling it explicitly here. Its split uses the same parameters
(`test_size=0.2, random_state=42, stratify=y`) as RQ1's holdout split, so
this does not introduce a second, inconsistent split -- it materializes
the same split as CSV files for direct inspection/reproducibility,
alongside the index-based split already used everywhere else.

In [ ]:
from preprocessing import run_screening_pipeline

run_screening_pipeline()
print("Populated data/model_ready/: X_train.csv, X_test.csv, y_train.csv, y_test.csv")


## Next steps

- Export screening results into `docs/data_dictionary_template.csv`
- Proceed to `02_modeling_rq1.ipynb` for baseline + imbalance-corrected models

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "01_data_screening"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"
live_export_available = False

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed. This only
    # works when there's an actual live frontend attached (i.e. you're running
    # this cell interactively yourself) -- it returns None instead of raising
    # when run unattended (e.g. via 00_run_all.ipynb's automated execution),
    # so that case is caught explicitly here rather than left to crash with a
    # raw TypeError, which used to make 00_run_all's --allow-errors flag mask
    # *real* failures elsewhere in the notebook, not just this expected one.
    from google.colab import _message
    response = _message.blocking_request('get_ipynb', timeout_sec=30)
    if response is not None:
        ipynb_content = response['ipynb']
        with open(export_path, 'w') as f:
            json.dump(ipynb_content, f)
        live_export_available = True
    else:
        print(
            "No live Colab frontend detected (expected when run via "
            "00_run_all.ipynb) -- skipping the live export. 00_run_all does its "
            "own separate HTML export against the already-executed file instead."
        )

if live_export_available or not IN_COLAB:
    html_output = f"{NOTEBOOK_NAME}.html"
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"Exported to {html_output}")

    if IN_COLAB and result.returncode == 0:
        from google.colab import files
        files.download(html_output)
